In [0]:
# 08_maintenance
# Periodic maintenance tasks for market-pulse-pipeline Delta tables.
# Run weekly via Lakeflow Job to keep tables optimised.

import sys
sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")

from market_pulse.config import (
    BRONZE_INGESTION_PATH,
    SILVER_DIM_STOCK_PATH,
    SILVER_FACT_PRICES_PATH,
    GOLD_DAILY_SUMMARY_PATH,
    GOLD_VOLUME_ANALYSIS_PATH,
    GOLD_MOVING_AVERAGES_PATH,
    GOLD_VOLATILITY_PATH,
    GOLD_STOCK_COMPARISON_PATH
)
from market_pulse.logger import get_logger
from market_pulse.config import PIPELINE_LOG_PATH

logger = get_logger(__name__, spark=spark, log_table_path=PIPELINE_LOG_PATH)

# Tables to optimise
TABLES = {
    "bronze_ingestion":   (BRONZE_INGESTION_PATH,        ["symbol", "trade_date"]),
    "silver_dim_stock":   (SILVER_DIM_STOCK_PATH,        ["symbol"]),
    "silver_fact_prices": (SILVER_FACT_PRICES_PATH,      ["symbol", "trade_date"]),
    "gold_daily_summary": (GOLD_DAILY_SUMMARY_PATH,      ["symbol", "trade_date"]),
    "gold_volume":        (GOLD_VOLUME_ANALYSIS_PATH,    ["symbol", "trade_date"]),
    "gold_moving_avg":    (GOLD_MOVING_AVERAGES_PATH,    ["symbol", "trade_date"]),
    "gold_volatility":    (GOLD_VOLATILITY_PATH,         ["symbol", "trade_date"]),
    "gold_comparison":    (GOLD_STOCK_COMPARISON_PATH,   ["symbol", "trade_date"]),
}

print("✅ Configuração carregada")
print(f"  Tables to optimise: {len(TABLES)}")

In [0]:
# Optimise all tables
for table_name, (path, zorder_cols) in TABLES.items():
    logger.info("optimizing", table=table_name)
    zorder_str = ", ".join(zorder_cols)
    spark.sql(f"OPTIMIZE delta.`{path}` ZORDER BY ({zorder_str})")
    logger.info("optimized", table=table_name)

logger.info("maintenance_complete", tables=len(TABLES))
print("✅ Maintenance complete")

In [0]:
# ─── Maintenance Report ───────────────────────────────────────────────────────
from pyspark.sql.functions import col, to_timestamp

print("📊 Maintenance Report")
print("─" * 40)

spark.read.format("delta").load(PIPELINE_LOG_PATH)\
    .filter(col("event").isin("optimizing", "optimized", "maintenance_complete"))\
    .orderBy("timestamp")\
    .select("timestamp", "event", "table")\
    .display()